In [1]:
!pip -q install transformers datasets accelerate scikit-learn

In [2]:
import numpy as np
import pandas as pd
import torch

from google.colab import drive

drive.mount("/content/drive")

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
GPU available: False


In [3]:
canonical_df = pd.read_pickle(
    "/content/drive/MyDrive/router_project_canonical.pkl"
)

splits = np.load(
    "/content/drive/MyDrive/router_split_indices.npz"
)

FINAL_THRESHOLD = float(
    np.load(
        "/content/drive/MyDrive/distilbert_final_threshold.npy"
    )[0]
)

train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
test_idx = splits["test_idx"]

print("Dataset:", canonical_df.shape)
print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

print("\nFROZEN DISTILBERT THRESHOLD:")
print(FINAL_THRESHOLD)

Dataset: (22962, 26)
Train: 18369
Validation: 2296
Test: 2297

FROZEN DISTILBERT THRESHOLD:
0.94


In [4]:
MODEL_COLS = [
    "WizardLM/WizardLM-13B-V1.2",
    "claude-instant-v1",
    "claude-v1",
    "claude-v2",
    "gpt-3.5-turbo-1106",
    "gpt-4-1106-preview",
    "meta/code-llama-instruct-34b-chat",
    "meta/llama-2-70b-chat",
    "mistralai/mistral-7b-chat",
    "mistralai/mixtral-8x7b-chat",
    "zero-one-ai/Yi-34B-Chat"
]

X = canonical_df[
    [
        "sample_id",
        "prompt",
        "domain",
        "eval_name"
    ]
].copy()

Y = pd.DataFrame({
    model:
    canonical_df[f"{model}|performance"]
    for model in MODEL_COLS
})

C = pd.DataFrame({
    model:
    canonical_df[f"{model}|cost"]
    for model in MODEL_COLS
})

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)

Y_train = Y.iloc[train_idx].reset_index(drop=True)
Y_test = Y.iloc[test_idx].reset_index(drop=True)

C_train = C.iloc[train_idx].reset_index(drop=True)
C_test = C.iloc[test_idx].reset_index(drop=True)

print("Test X:", X_test.shape)
print("Test Y:", Y_test.shape)
print("Test C:", C_test.shape)

Test X: (2297, 4)
Test Y: (2297, 11)
Test C: (2297, 11)


In [5]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODEL_PATH = (
    "/content/drive/MyDrive/"
    "distilbert_llm_router"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

print("Model loaded.")

print(
    "Outputs:",
    model.config.num_labels
)

print(
    "Problem type:",
    model.config.problem_type
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded.
Outputs: 11
Problem type: regression


In [6]:
from datasets import Dataset, Features, Value, Sequence

features = Features({
    "text": Value("string"),
    "labels": Sequence(
        Value("float32"),
        length=11
    )
})

test_ds = Dataset.from_dict(
    {
        "text":
            X_test["prompt"].astype(str).tolist(),

        "labels":
            Y_test.astype(
                "float32"
            ).values.tolist()
    },
    features=features
)

print(test_ds)

Dataset({
    features: ['text', 'labels'],
    num_rows: 2297
})


In [7]:
MAX_LENGTH = 384

def tokenize_batch(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

test_tokenized = test_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

if "token_type_ids" in test_tokenized.column_names:
    test_tokenized = test_tokenized.remove_columns(
        ["token_type_ids"]
    )

print(test_tokenized)

Map:   0%|          | 0/2297 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 2297
})


In [8]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

eval_args = TrainingArguments(
    output_dir="/content/final_test_eval",

    per_device_eval_batch_size=32,

    fp16=True,

    report_to="none"
)

test_trainer = Trainer(
    model=model,
    args=eval_args,
    data_collator=data_collator,
    processing_class=tokenizer
)

test_output = test_trainer.predict(
    test_tokenized
)

distilbert_test_pred = (
    test_output.predictions
)

print(
    "Prediction shape:",
    distilbert_test_pred.shape
)

print(
    "Raw range:",
    distilbert_test_pred.min(),
    "to",
    distilbert_test_pred.max()
)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Prediction shape: (2297, 11)
Raw range: -0.080555856 to 1.0844014


In [9]:
distilbert_test_pred = np.clip(
    distilbert_test_pred,
    0.0,
    1.0
)

print(
    "Clipped range:",
    distilbert_test_pred.min(),
    "to",
    distilbert_test_pred.max()
)

Clipped range: 0.0 to 1.0


In [10]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)

test_mse = mean_squared_error(
    Y_test,
    distilbert_test_pred
)

test_mae = mean_absolute_error(
    Y_test,
    distilbert_test_pred
)

print(
    "FINAL TEST MSE:",
    test_mse
)

print(
    "FINAL TEST MAE:",
    test_mae
)

FINAL TEST MSE: 0.12370815873146057
FINAL TEST MAE: 0.2628079652786255


In [11]:
mean_train_cost = (
    C_train.mean().to_numpy()
)

def route_predictions(
    predictions,
    model_costs,
    threshold
):

    choices = []

    cost_order = np.argsort(
        model_costs
    )

    for scores in predictions:

        eligible = [
            k
            for k in cost_order
            if scores[k] >= threshold
        ]

        if len(eligible) > 0:

            selected = eligible[0]

        else:

            selected = int(
                np.argmax(scores)
            )

        choices.append(selected)

    return np.array(choices)

In [12]:
test_choices = route_predictions(
    distilbert_test_pred,
    mean_train_cost,
    FINAL_THRESHOLD
)

rows = np.arange(
    len(test_choices)
)

Y_test_np = Y_test.to_numpy()
C_test_np = C_test.to_numpy()

router_test_performance = (
    Y_test_np[
        rows,
        test_choices
    ]
)

router_test_cost = (
    C_test_np[
        rows,
        test_choices
    ]
)

print(
    "FINAL ROUTER PERFORMANCE:",
    router_test_performance.mean()
)

print(
    "FINAL ROUTER COST:",
    router_test_cost.mean()
)

FINAL ROUTER PERFORMANCE: 0.7604484
FINAL ROUTER COST: 0.0028767393


In [13]:
BEST_FIXED_MODEL = (
    "gpt-4-1106-preview"
)

CHEAPEST_FIXED_MODEL = (
    "mistralai/mistral-7b-chat"
)

In [14]:
best_fixed_test_performance = (
    Y_test[
        BEST_FIXED_MODEL
    ].mean()
)

best_fixed_test_cost = (
    C_test[
        BEST_FIXED_MODEL
    ].mean()
)

cheapest_test_performance = (
    Y_test[
        CHEAPEST_FIXED_MODEL
    ].mean()
)

cheapest_test_cost = (
    C_test[
        CHEAPEST_FIXED_MODEL
    ].mean()
)

print("GPT-4 fixed:")
print(
    "Performance:",
    best_fixed_test_performance
)
print(
    "Cost:",
    best_fixed_test_cost
)

print("\nCheapest fixed:")
print(
    "Performance:",
    cheapest_test_performance
)
print(
    "Cost:",
    cheapest_test_cost
)

GPT-4 fixed:
Performance: 0.76806706
Cost: 0.0037382713

Cheapest fixed:
Performance: 0.32270354
Cost: 4.8630216e-05


In [15]:
oracle_scores = []
oracle_costs = []

for i in range(len(Y_test_np)):

    scores = Y_test_np[i]
    costs = C_test_np[i]

    best_score = scores.max()

    candidates = np.where(
        scores == best_score
    )[0]

    selected = candidates[
        np.argmin(
            costs[candidates]
        )
    ]

    oracle_scores.append(
        scores[selected]
    )

    oracle_costs.append(
        costs[selected]
    )

oracle_test_performance = np.mean(
    oracle_scores
)

oracle_test_cost = np.mean(
    oracle_costs
)

print(
    "Oracle test performance:",
    oracle_test_performance
)

print(
    "Oracle test cost:",
    oracle_test_cost
)

Oracle test performance: 0.8886591
Oracle test cost: 0.000263036


In [16]:
final_performance = (
    router_test_performance.mean()
)

final_cost = (
    router_test_cost.mean()
)

performance_gap = (
    best_fixed_test_performance
    - final_performance
) * 100

cost_saving = (
    1
    - final_cost /
      best_fixed_test_cost
) * 100

print(
    "===== FINAL PROJECT RESULT ====="
)

print(
    "Frozen threshold:",
    FINAL_THRESHOLD
)

print(
    "GPT-4 performance:",
    best_fixed_test_performance
)

print(
    "Router performance:",
    final_performance
)

print(
    "Performance gap:",
    performance_gap,
    "percentage points"
)

print()

print(
    "GPT-4 cost:",
    best_fixed_test_cost
)

print(
    "Router cost:",
    final_cost
)

print(
    "Cost saving:",
    cost_saving,
    "%"
)

print()

if performance_gap <= 1.0:

    print(
        "MAIN HYPOTHESIS SATISFIED"
    )

else:

    print(
        "MAIN HYPOTHESIS NOT SATISFIED"
    )

===== FINAL PROJECT RESULT =====
Frozen threshold: 0.94
GPT-4 performance: 0.76806706
Router performance: 0.7604484
Performance gap: 0.76186657 percentage points

GPT-4 cost: 0.0037382713
Router cost: 0.0028767393
Cost saving: 23.046267 %

MAIN HYPOTHESIS SATISFIED


In [17]:
final_comparison = pd.DataFrame([
    {
        "System":
            "Cheapest Fixed",
        "Performance":
            cheapest_test_performance,
        "Cost":
            cheapest_test_cost
    },

    {
        "System":
            "GPT-4 Fixed",
        "Performance":
            best_fixed_test_performance,
        "Cost":
            best_fixed_test_cost
    },

    {
        "System":
            "DistilBERT Router",
        "Performance":
            final_performance,
        "Cost":
            final_cost
    },

    {
        "System":
            "Oracle",
        "Performance":
            oracle_test_performance,
        "Cost":
            oracle_test_cost
    }
])

display(final_comparison)

,System,Performance,Cost
0,Cheapest Fixed,0.322704,0.000049
1,GPT-4 Fixed,0.768067,0.003738
2,DistilBERT Router,0.760448,0.002877
3,Oracle,0.888659,0.000263


In [18]:
final_comparison.to_csv(
    "/content/drive/MyDrive/"
    "final_test_comparison.csv",
    index=False
)

np.save(
    "/content/drive/MyDrive/"
    "distilbert_test_predictions.npy",
    distilbert_test_pred
)

np.save(
    "/content/drive/MyDrive/"
    "distilbert_test_choices.npy",
    test_choices
)

print("Final test artifacts saved.")

Final test artifacts saved.
